In [ ]:
'''# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session'''

### Introduction

In this experiment, we extend the GCN architecture from 2 layers to 4 layers to study the effect of model depth on performance. While additional layers enable nodes to capture information from larger neighborhoods, they can also lead to **oversmoothing**, where node embeddings become indistinguishable across classes.

In [2]:
!pip install torch-geometric
#!pip install pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv

In [3]:
import torch
import torch_geometric

print(torch.__version__)
print(torch_geometric.__version__)

2.10.0+cu128
2.7.0


### Loading Amazon Computer Dataset

In [4]:
from torch_geometric.datasets import Amazon
dataset=Amazon(root='/kaggle/working/Amazon',name='Computers')
data=dataset[0]
print(data)

Processing...


Data(x=[13752, 767], edge_index=[2, 491722], y=[13752])


Done!


In [5]:
print(data.x.shape)
print(data.edge_index.shape)
print(data.y.shape)
print(data.y.unique())


torch.Size([13752, 767])
torch.Size([2, 491722])
torch.Size([13752])
tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


###  Loading the train test validation Split(70% train 15% validation 15% test)



In [ ]:


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_idx = torch.load("/kaggle/input/datasets/mrxn251/gnn-splits/train_idx.pt")
val_idx = torch.load("/kaggle/input/datasets/mrxn251/gnn-splits/val_idx.pt")
test_idx = torch.load("/kaggle/input/datasets/mrxn251/gnn-splits/test_idx.pt")
train_idx = train_idx.to(device)
val_idx = val_idx.to(device)
test_idx = test_idx.to(device)

In [7]:
print(len(train_idx), len(val_idx), len(test_idx))

9626 2063 2063


In [8]:
#moving data to GPU
data = data.to(device)

### MINI BATCHING

In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)

In [ ]:
'''# Install dependencies from source (this may take a few minutes)
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.10.0+cu128.html
!pip install torch-geometric

# IMPORTANT: Check if it actually installed correctly
try:
    import pyg_lib
    import torch_sparse
    print("Success: Libraries are ready!")
except ImportError:
    print("Binaries not found, trying an alternative install...")
    # If the above fails, you can try installing without the '-f' link to force a local build
    !pip install pyg-lib torch-scatter torch-sparse torch-cluster -U'''

In [9]:
import torch
import torch_sparse
import pyg_lib

print(f"PyG-Lib Version: {pyg_lib.__version__}")
print(f"Torch-Sparse Version: {torch_sparse.__version__}")

PyG-Lib Version: 0.6.0+pt210cu128
Torch-Sparse Version: 0.6.18+pt210cu128


In [10]:
from torch_geometric.loader import NeighborLoader

###  defining GCN model architecture with 4 layers 

In [11]:
from torch_geometric.nn import GCNConv
import torch.nn.functional as F
import torch

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels)
        self.conv4 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        x = self.conv3(x, edge_index)
        x = F.relu(x)

        x = self.conv4(x, edge_index)

        return x

### Training function to perform 1 epoch of training 

In [12]:
def train(model,loader,optimizer,device):
    model.train()
    total_loss=0
    for batch in loader:
        batch=batch.to(device)
        optimizer.zero_grad()
        out=model(batch.x,batch.edge_index)
        loss=F.cross_entropy(out[:batch.batch_size],batch.y[:batch.batch_size])
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    return total_loss/len(loader)

### Evaluation function to test performance on validation set

In [13]:
@torch.no_grad()
def evaluate(model,loader,device):
    model.eval()
    correct=0
    total=0
    for batch in loader:
        batch=batch.to(device)
        out=model(batch.x,batch.edge_index)
        pred=out[:batch.batch_size].argmax(dim=1)
        correct+=(pred==batch.y[:batch.batch_size]).sum().item()
        total+=batch.batch_size
    return correct/total

In [19]:
import random
import numpy as np
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    

### Testing Function to evaluate model performance on test dataset 

In [20]:
from sklearn.metrics import f1_score

@torch.no_grad()
def test_with_f1(model, loader, device):
    model.eval()
    
    all_preds = []
    all_labels = []

    for batch in loader:
        batch = batch.to(device)

        out = model(batch.x, batch.edge_index)
        pred = out[:batch.batch_size].argmax(dim=1)

        all_preds.append(pred.cpu())
        all_labels.append(batch.y[:batch.batch_size].cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    acc = (all_preds == all_labels).mean()
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    micro_f1 = f1_score(all_labels, all_preds, average='micro')

    return acc, macro_f1, micro_f1

### Experiment Pipeline

This function manages the end-to-end training and evaluation process.

- Initializes the model using the given configuration
- Constructs NeighborLoaders for mini-batch training
- Trains the model across epochs with early stopping based on validation performance
- Saves the best-performing model
- Evaluates final performance on the test set using Accuracy and F1 scores

In [23]:
def run_experiment(config, seed=0, run_test=False):
    set_seed(seed)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    #epochs=4
    epochs = 50 if not run_test else 100
    model = GCN(
        data.num_features,
        config["hidden_dim"],
        dataset.num_classes
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])


    train_loader = NeighborLoader(
        data,
        input_nodes=train_idx,
        num_neighbors=config["num_neighbors"],
        batch_size=128,
        shuffle=True
    )

    val_loader = NeighborLoader(
        data,
        input_nodes=val_idx,
        num_neighbors=config["num_neighbors"],
        batch_size=256,
        shuffle=False
    )

    best_val = 0
    patience = 10
    counter = 0
    save_model=run_test 

    for epoch in range(1,epochs+1):
        loss = train(model, train_loader, optimizer, device)
        val_acc = evaluate(model, val_loader, device)

        if val_acc > best_val:
            best_val = val_acc
            counter = 0
            if save_model:
             torch.save(model.state_dict(), f"/kaggle/working/best_modelGCN_{seed}.pt")
        else:
            counter += 1

        if counter >= patience:
            break
    if not run_test:
        return best_val

    # load best model
    model.load_state_dict(torch.load(f"/kaggle/working/best_modelGCN_{seed}.pt", map_location=device))

    if not run_test:
        return best_val

    test_loader = NeighborLoader(
        data,
        input_nodes=test_idx,
        num_neighbors=config["num_neighbors"],
        batch_size=256,
        shuffle=False
    )

    acc, macro_f1, micro_f1 = test_with_f1(model, test_loader, device)

    return acc, macro_f1,micro_f1
    

    

The model is initialized using the best hyperparameters obtained from the previous experiment.

To ensure a fair comparison, hyperparameter tuning is not repeated, and the same configuration is used while varying only the model depth.

In [ ]:
best_config= {'hidden_dim': 32, 'lr': 0.001, 'num_neighbors': [20, 10]}

### Final Training and evaluation of model across 10 different seed values 

In [27]:
#Final Training
acc_list = []
macro_f1_list = []
micro_f1_list=[]
for seed in range(10):
    acc, macro_f1,micro_f1  = run_experiment(best_config, seed=seed, run_test=True)
    acc_list.append(acc)
    macro_f1_list.append(macro_f1)
    micro_f1_list.append(micro_f1)

    print(f"Seed {seed}: Acc={acc:.4f}, macro_F1={macro_f1:.4f},micro_F1={micro_f1:.4f}")

Seed 0: Acc=0.8851, macro_F1=0.8767,micro_F1=0.8851
Seed 1: Acc=0.8871, macro_F1=0.8817,micro_F1=0.8871
Seed 2: Acc=0.8827, macro_F1=0.8779,micro_F1=0.8827
Seed 3: Acc=0.8871, macro_F1=0.8763,micro_F1=0.8871
Seed 4: Acc=0.8890, macro_F1=0.8813,micro_F1=0.8890
Seed 5: Acc=0.8963, macro_F1=0.8933,micro_F1=0.8963
Seed 6: Acc=0.8861, macro_F1=0.8861,micro_F1=0.8861
Seed 7: Acc=0.8871, macro_F1=0.8766,micro_F1=0.8871
Seed 8: Acc=0.8885, macro_F1=0.8819,micro_F1=0.8885
Seed 9: Acc=0.8914, macro_F1=0.8840,micro_F1=0.8914


### Final Results 

In [28]:
import numpy as np

acc_mean = np.mean(acc_list)
acc_std = np.std(acc_list)

macro_f1_mean = np.mean(macro_f1_list)
macro_f1_std = np.std(macro_f1_list)

micro_f1_mean = np.mean(micro_f1_list)
micro_f1_std = np.std(micro_f1_list)

print(f"\nFinal Results:")
print(f"Accuracy: {acc_mean:.4f} ± {acc_std:.4f}")
print(f"Macro F1: {macro_f1_mean:.4f} ± {macro_f1_std:.4f}")
print(f"Micro F1: {micro_f1_mean:.4f} ± {micro_f1_std:.4f}")


Final Results:
Accuracy: 0.8880 ± 0.0035
Macro F1: 0.8816 ± 0.0050
Micro F1: 0.8880 ± 0.0035
